# 4. Bayesian Modeling

This notebook implements a Bayesian Neural Network (BNN) using TensorFlow Probability. We mirror the deterministic architecture exactly (128→64→32, dropout 0.2, ReLU, same 5 segments), replacing standard Dense layers with probabilistic `DenseFlipout` layers.

Unlike the deterministic model, the BNN treats weights as distributions rather than fixed values. Each forward pass samples different weights from the learned posterior, producing a distribution over predictions rather than a single point estimate. This gives us both a mean prediction and a calibrated uncertainty estimate per input.

**Key Technical Details:**
- **Loss:** Negative Log-Likelihood (NLL) + KL Divergence, equivalent to maximizing the ELBO.
- **Layers:** `DenseFlipout` — decorrelates gradients within a mini-batch for more stable training.
- **KL Scaling:** KL term is scaled by $1/N$ to balance the likelihood correctly across mini-batches.
- **Inference:** Monte Carlo sampling (100 forward passes) to estimate mean and uncertainty.

**Steps:**
1. **Data Loading and Splitting:** Load the preprocessed dataset and split into train/test sets.
2. **Model Design:** Define the probabilistic architecture and training utilities.
3. **Training:** Train one Bayesian model per segment using variational inference.
4. **Evaluation:** Monte Carlo inference to compute mean predictions and uncertainty.
5. **Artifact Export:** Save weights per segment for downstream use.

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import tensorflow_probability as tfp
import tf_keras as keras
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import os
import time
import warnings

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

tfd  = tfp.distributions
tfpl = tfp.layers

GLOBAL_SEED = 42
tf.random.set_seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

## Data Loading and Splitting

In [ ]:
df = pd.read_csv('../../data/madrid/listings_experimental_processed.csv')

X = df.drop('price', axis=1).values.astype('float32')
y = df['price'].values.astype('float32')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train features shape: {X_train.shape}")
print(f"Test features shape: {X_test.shape}")

## Bayesian Model Design

We define the probabilistic counterpart of the deterministic architecture. Each Dense layer is replaced by a `DenseFlipout` layer that maintains a distribution over its weights instead of a fixed value. The output layer produces a Normal distribution $\mathcal{N}(\mu, \sigma)$, where $\sigma$ is guaranteed positive via `softplus`.

In [ ]:
SEGMENTS = [
    {'name': 'Low',         'min': 0,   'max': 79},
    {'name': 'Low_Medium',  'min': 79,  'max': 110},
    {'name': 'Medium',      'min': 110, 'max': 150},
    {'name': 'Medium_High', 'min': 150, 'max': 193},
    {'name': 'High',        'min': 193, 'max': 249}
]

def build_bayesian_model(input_dim, n_train):
    kl_weight    = 1.0 / n_train
    divergence_fn = lambda q, p, _: tfd.kl_divergence(q, p) * kl_weight

    model = keras.Sequential(name='Bayesian_Model')
    model.add(keras.layers.Input(shape=(input_dim,)))

    for units in [128, 64, 32]:
        model.add(tfpl.DenseFlipout(units, activation='relu', kernel_divergence_fn=divergence_fn))

    model.add(tfpl.DenseFlipout(2, activation=None, kernel_divergence_fn=divergence_fn))
    model.add(tfpl.DistributionLambda(
        lambda t: tfd.Normal(
            loc=t[..., :1],
            scale=1e-3 + tf.math.softplus(0.01 * t[..., 1:])
        )
    ))

    def neg_log_likelihood(y_true, y_pred):
        return -y_pred.log_prob(y_true)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss=neg_log_likelihood
    )
    return model

def predict_mc(model, X, n_iter=200):
    X_tensor = np.array(X, dtype=np.float32)
    samples  = np.array([model(X_tensor).mean().numpy() for _ in range(n_iter)])
    return samples.mean(axis=0).flatten(), samples.std(axis=0).flatten()

## Training

For each of the 5 price segments, we filter the training set by price range and train an independent Bayesian model using variational inference.

In [ ]:
bayesian_models = {}
training_records = []

RESULTS_DIR = '../training'
os.makedirs(RESULTS_DIR, exist_ok=True)

for seg in SEGMENTS:
    name = seg['name']
    print(f"Training {name} (€{seg['min']}-{seg['max']})")

    mask_tr = (y_train > seg['min']) & (y_train <= seg['max'])
    mask_te = (y_test  > seg['min']) & (y_test  <= seg['max'])

    x_tr, y_tr = X_train[mask_tr], y_train[mask_tr]
    x_te, y_te = X_test[mask_te],  y_test[mask_te]

    model = build_bayesian_model(x_tr.shape[1], len(x_tr))

    start = time.time()
    history = model.fit(
        x_tr, y_tr,
        validation_data=(x_te, y_te),
        epochs=1000, batch_size=32,
        callbacks=[
            keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
        ],
        verbose=0
    )
    training_time = time.time() - start

    y_pred_mean, y_pred_std = predict_mc(model, x_te)
    mae  = float(np.mean(np.abs(y_te - y_pred_mean)))
    rmse = float(np.sqrt(np.mean((y_te - y_pred_mean) ** 2)))
    mape = float(np.mean(np.abs((y_te - y_pred_mean) / y_te)) * 100)
    unc  = float(np.mean(y_pred_std))

    print(f"  MAE: €{mae:.2f} | MAPE: {mape:.2f}% | Avg uncertainty: €{unc:.2f} | Time: {training_time:.1f}s")

    bayesian_models[name] = model
    training_records.append({
        'segment': name,
        'price_min': seg['min'], 'price_max': seg['max'],
        'n_train': len(y_tr), 'n_test': len(y_te),
        'test_mae': round(mae, 4), 'test_rmse': round(rmse, 4),
        'test_mape': round(mape, 4), 'avg_uncertainty': round(unc, 4),
        'epochs_trained': len(history.history['loss'])
    })

pd.DataFrame(training_records).to_csv(f'{RESULTS_DIR}/bayesian_metrics.csv', index=False)
print(f"\nResults saved to: {RESULTS_DIR}/bayesian.csv")

## Probabilistic Evaluation

We visualize two aspects of the Bayesian output for each segment: prediction intervals (mean ± 2σ) and the relationship between predicted price and uncertainty (heteroscedasticity).

In [ ]:
IMG_DIR = '../training'
os.makedirs(IMG_DIR, exist_ok=True)

for seg in SEGMENTS:
    name = seg['name']
    mask_te = (y_test > seg['min']) & (y_test <= seg['max'])
    x_te = X_test[mask_te]
    y_te = y_test[mask_te]

    y_pred_mean, y_pred_std = predict_mc(bayesian_models[name], x_te)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Segment: {name} (€{seg["min"]}-{seg["max"]})', fontsize=14, fontweight='bold')

    # Plot 1: Prediction intervals
    ax = axes[0]
    idx = np.random.choice(len(y_te), min(60, len(y_te)), replace=False)
    ax.errorbar(y_te[idx], y_pred_mean[idx], yerr=2*y_pred_std[idx],
                fmt='o', ecolor='gray', alpha=0.5, label='Uncertainty (2σ)')
    ax.scatter(y_te[idx], y_pred_mean[idx], c='steelblue', alpha=0.7, label='Mean prediction')
    lims = [seg['min'], seg['max']]
    ax.plot(lims, lims, 'r--', label='Perfect prediction')
    ax.set_xlabel('Actual price (€)')
    ax.set_ylabel('Predicted price (€)')
    ax.set_title('Prediction intervals')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # Plot 2: Heteroscedasticity
    ax = axes[1]
    ax.scatter(y_pred_mean, y_pred_std, alpha=0.3, c='purple')
    ax.set_xlabel('Predicted price (€)')
    ax.set_ylabel('Uncertainty (σ)')
    ax.set_title('Heteroscedasticity check')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{IMG_DIR}/bayesian_{name.lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()

## Artifact Export

Due to the complex nature of TensorFlow Probability layers (which involve custom divergence functions and probabilistic weights), the standard Keras `save_model` format is often insufficient or problematic.

Instead, we serialize the model by saving only the weights. To restore the model later, the architecture must be redefined exactly as above before loading these weights.

In [ ]:
BAY_DIR = '../../artifacts/bayesian'
os.makedirs(BAY_DIR, exist_ok=True)

for name, model in bayesian_models.items():
    weights_path = os.path.join(BAY_DIR, f'segment_{name}_weights.h5')
    model.save_weights(weights_path)

print(f"Exported {len(bayesian_models)} Bayesian segment models to: {BAY_DIR}")